# Лабораторная работа 13: Gradient Boosting и Random Forest
Выполнение заданий по градиентному бустингу и случайному лесу на данных gbm-data.csv.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import log_loss

1. Загрузите выборку из файла gbm-data.csv с помощью pandas и преобразуйте ее в массив numpy (параметр values у датафрейма). В первой колонке файла с данными записано, была или нет реакция. Все остальные колонки (d1 - d1776) содержат различные характеристики молекулы, такие как размер, форма и т.д. Разбейте выборку на обучающую и тестовую, используя функцию train_test_split с параметрами test_size = 0.8 и random_state = 241.

In [2]:
data_mol = pd.read_csv('gbm-data.csv')
data_mol

,Activity,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,D1767,D1768,D1769,D1770,D1771,D1772,D1773,D1774,D1775,D1776
0,1,0.000000,0.497009,0.10,0.0,0.132956,0.678031,0.273166,0.585445,0.743663,...,0,0,0,0,0,0,0,0,0,0
1,1,0.366667,0.606291,0.05,0.0,0.111209,0.803455,0.106105,0.411754,0.836582,...,1,1,1,1,0,1,0,0,1,0
2,1,0.033300,0.480124,0.00,0.0,0.209791,0.610350,0.356453,0.517720,0.679051,...,0,0,0,0,0,0,0,0,0,0
3,1,0.000000,0.538825,0.00,0.5,0.196344,0.724230,0.235606,0.288764,0.805110,...,0,0,0,0,0,0,0,0,0,0
4,0,0.100000,0.517794,0.00,0.0,0.494734,0.781422,0.154361,0.303809,0.812646,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3746,1,0.033300,0.506409,0.10,0.0,0.209887,0.633426,0.297659,0.376124,0.727093,...,0,0,0,0,0,0,0,0,0,0
3747,1,0.133333,0.651023,0.15,0.0,0.151154,0.766505,0.170876,0.404546,0.787935,...,0,0,1,0,1,0,1,0,0,0
3748,0,0.200000,0.520564,0.00,0.0,0.179949,0.768785,0.177341,0.471179,0.872241,...,0,0,0,0,0,0,0,0,0,0
3749,1,0.100000,0.765646,0.00,0.0,0.536954,0.634936,0.342713,0.447162,0.672689,...,0,0,0,0,0,0,0,0,0,0


In [3]:
data = data_mol.values
X = data[:, 1:]
y = data[:, 0]

X.shape

(3751, 1776)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.8, random_state=241)

2. Обучите GradientBoostingClassifier с параметрами n_estimators=250, verbose=True, random_state=241 и для каждого значения learning_rate из списка [1, 0.5, 0.3, 0.2, 0.1] проделайте следующее:
- Используйте метод staged_decision_function для предсказания качества на обучающей и тестовой выборке на каждой итерации.
- Преобразуйте полученное предсказание по формуле \( \frac{1}{1+e^{-y_{pred}}} \), где \( y_{pred} \) — предсказанное значение.
- Вычислите и постройте график значений log-loss на обучающей и тестовой выборках, а также найдите минимальное значение метрики и номер итерации, на которой оно достигается.

In [5]:

LR = [1, 0.5, 0.3, 0.2, 0.1]
res_loss = {}
for lr in LR:
    model = GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=lr,
        random_state=241,
        verbose=False
    )
    
    model.fit(X_train, y_train)
    
    train_loss = []
    test_loss = []
    
    for y_pred_train, y_pred_test in zip(
        model.staged_decision_function(X_train),
        model.staged_decision_function(X_test)
    ):
        prob_train = 1 / (1 + np.exp(-y_pred_train))
        prob_test = 1 / (1 + np.exp(-y_pred_test))
        
        train_loss.append(log_loss(y_train, prob_train))
        test_loss.append(log_loss(y_test, prob_test))
    
    min_loss_test = min(test_loss)
    best_iter_test = np.argmin(test_loss)

    min_loss_train = min(train_loss)
    best_iter_train = np.argmin(train_loss)

    
    res_loss[lr] = [(int(best_iter_train), min_loss_train), (int(best_iter_test), min_loss_test)]
    
    print(f"learning_rate={lr}")
    print(f"{res_loss[lr]}")
    print()
    
    # график
    plt.plot(test_loss, label=f"test lr={lr}")
    plt.plot(train_loss, linestyle="--", label=f"train lr={lr}")
plt.xlabel("Iteration")
plt.ylabel("Log-loss")
plt.legend()
plt.show()


'\nLR = [1, 0.5, 0.3, 0.2, 0.1]\nres_loss = {}\nfor lr in LR:\n    model = GradientBoostingClassifier(\n        n_estimators=250,\n        learning_rate=lr,\n        random_state=241,\n        verbose=False\n    )\n    \n    model.fit(X_train, y_train)\n    \n    train_loss = []\n    test_loss = []\n    \n    for y_pred_train, y_pred_test in zip(\n        model.staged_decision_function(X_train),\n        model.staged_decision_function(X_test)\n    ):\n        prob_train = 1 / (1 + np.exp(-y_pred_train))\n        prob_test = 1 / (1 + np.exp(-y_pred_test))\n        \n        train_loss.append(log_loss(y_train, prob_train))\n        test_loss.append(log_loss(y_test, prob_test))\n    \n    min_loss_test = min(test_loss)\n    best_iter_test = np.argmin(test_loss)\n\n    min_loss_train = min(train_loss)\n    best_iter_train = np.argmin(train_loss)\n\n    \n    res_loss[lr] = [(int(best_iter_train), min_loss_train), (int(best_iter_test), min_loss_test)]\n    \n    print(f"learning_rate={lr}")

3. Как можно охарактеризовать график качества на тестовой выборке, начиная с некоторой итерации: переобучение (overfitting) или недообучение (underfitting)? В ответе укажите одно из слов overfitting либо underfitting.

In [6]:
ans_1 = 'overfiting'
with open('1.txt', 'w') as file:
    file.write(f'{ans_1}')

4. Приведите минимальное значение log-loss на тестовой выборке и номер итерации, на котором оно достигается, при learning_rate = 0.2.

In [7]:
ans_2 = res_loss[0.2][1][0], res_loss[0.2][1][1]
with open('2.txt', 'w') as file:
    file.write(f'{ans_2[0]} {ans_2[1]:.2f}')

NameError: name 'res_loss' is not defined

5. На этих же данных обучите RandomForestClassifier с количеством деревьев, равным количеству итераций, на котором достигается наилучшее качество у градиентного бустинга из предыдущего пункта, random_state=241 и остальными параметрами по умолчанию. Какое значение log-loss на тесте получается у этого случайного леса? (Не забывайте, что предсказания нужно получать с помощью функции predict_proba. В данном случае брать сигмоиду от оценки вероятности класса не нужно.)

In [8]:
rf = RandomForestClassifier(random_state=241, n_estimators=res_loss[0.2][1][0]).fit(X_train, y_train)
pred_rf = rf.predict_proba(X_test)
ans_3 = log_loss(y_test, pred_rf)

with open('3.txt', 'w') as file:
    file.write(f'{ans_3:.2f}')

NameError: name 'res_loss' is not defined